# Navegador notebooks

In [ ]:
# Celda 1: Imports
from pathlib import Path
import pandas as pd
import json
from IPython.display import display, HTML, Markdown

try:
    import google.colab
    IN_COLAB = True
    PROJECT_ROOT = Path('/content/Estrategias-No-Convencionales')
except ImportError:
    IN_COLAB = False
    PROJECT_ROOT = Path('.')

NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'

print(f"📂 Proyecto: {PROJECT_ROOT}")
print(f" Notebooks: {NOTEBOOKS_DIR}")
print(f"✅ Existe: {NOTEBOOKS_DIR.exists()}")

# Celda 2: Escaneo automático
def escanear_notebooks():
    if not NOTEBOOKS_DIR.exists():
        print(f"ERROR: No existe {NOTEBOOKS_DIR}")
        return []
    
    notebooks_encontrados = []
    
    for bloque_dir in sorted(NOTEBOOKS_DIR.glob('Bloque_*')):
        if not bloque_dir.is_dir():
            continue
        
        bloque_nombre = bloque_dir.name
        
        for ipynb_path in sorted(bloque_dir.glob('*.ipynb')):
            if ipynb_path.name.startswith('.') or '.ipynb_checkpoints' in str(ipynb_path):
                continue
            
            titulo = ipynb_path.stem
            descripcion = ""
            
            try:
                with open(ipynb_path, 'r', encoding='utf-8') as f:
                    nb = json.load(f)
                
                for cell in nb.get('cells', []):
                    if cell.get('cell_type') == 'markdown':
                        source = ''.join(cell.get('source', []))
                        if source.strip():
                            for linea in source.split('\n'):
                                if linea.startswith('#'):
                                    titulo = linea.lstrip('# ').strip()
                                    break
                            lineas = source.split('\n')
                            desc_lineas = []
                            capturando = False
                            for linea in lineas:
                                if linea.startswith('#'):
                                    capturando = True
                                    continue
                                if capturando and linea.strip() and not linea.startswith('**'):
                                    desc_lineas.append(linea.strip())
                                if len(desc_lineas) >= 2:
                                    break
                            descripcion = ' '.join(desc_lineas)[:200]
                            break
            except Exception as e:
                descripcion = f"(Error leyendo: {str(e)[:50]})"
            
            notebooks_encontrados.append({
                'titulo': titulo,
                'archivo': ipynb_path.name,
                'ruta_completa': ipynb_path,
                'ruta_relativa': ipynb_path.relative_to(PROJECT_ROOT),
                'bloque': bloque_nombre,
                'descripcion': descripcion
            })
    
    return notebooks_encontrados

lista_notebooks = escanear_notebooks()
df_notebooks = pd.DataFrame(lista_notebooks)

print(f"✅ Se encontraron {len(df_notebooks)} notebooks organizados por bloques\n")
print("Distribución por bloque:")
for bloque in sorted(df_notebooks['bloque'].unique()):
    count = len(df_notebooks[df_notebooks['bloque'] == bloque])
    print(f"  • {bloque}: {count} notebooks")

# Celda 3: Menú visual interactivo con botones "Abrir en Colab"
def mostrar_notebooks(filtro_bloque=None, filtro_busqueda=None):
    df = df_notebooks.copy()
    
    if filtro_bloque and filtro_bloque != 'Todos':
        df = df[df['bloque'] == filtro_bloque]
    
    if filtro_busqueda:
        mask = (
            df['titulo'].str.contains(filtro_busqueda, case=False, na=False) |
            df['descripcion'].str.contains(filtro_busqueda, case=False, na=False)
        )
        df = df[mask]
    
    html = """
    <style>
        .nb-card {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 20px;
            margin: 15px 0;
            border-radius: 10px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
            transition: transform 0.3s;
        }
        .nb-card:hover { transform: translateY(-3px); box-shadow: 0 8px 12px rgba(0,0,0,0.2); }
        .nb-titulo { font-size: 1.3em; font-weight: bold; margin-bottom: 8px; }
        .nb-bloque { font-size: 0.85em; opacity: 0.9; background: rgba(255,255,255,0.2); padding: 3px 10px; border-radius: 12px; display: inline-block; margin-bottom: 10px; }
        .nb-descripcion { font-size: 0.95em; line-height: 1.5; margin-bottom: 15px; }
        .nb-archivo { font-size: 0.8em; color: #e0e0e0; font-family: monospace; margin-bottom: 12px; }
        .btn { display: inline-block; padding: 8px 16px; border-radius: 5px; text-decoration: none; margin-right: 8px; font-weight: bold; font-size: 0.9em; cursor: pointer; border: none; }
        .btn-colab { background: #f9ab00; color: white; }
        .btn-colab:hover { background: #e09800; }
        .btn-ver { background: #4CAF50; color: white; }
        .btn-ver:hover { background: #45a049; }
        h2 { color: #2c3e50; border-bottom: 3px solid #667eea; padding-bottom: 10px; }
    </style>
    """
    
    html += f"<h2>📓 Notebooks del Proyecto ({len(df)} encontrados)</h2>"
    
    if len(df) == 0:
        html += "<p>No se encontraron notebooks con los filtros aplicados.</p>"
    
    # Reemplaza TU_USUARIO por tu usuario real de GitHub
    GITHUB_USER = "akitxu"
    GITHUB_REPO = "Estrategias-No-Convencionales"
    
    for _, row in df.iterrows():
        ruta_rel = str(row['ruta_relativa']).replace(' ', '%20')
        
        # Enlace directo a Colab (abre el notebook en Colab con el repo montado)
        colab_url = f"https://colab.research.google.com/github/{GITHUB_USER}/{GITHUB_REPO}/blob/main/{ruta_rel}"
        
        btn_colab = f'<a href="{colab_url}" target="_blank" class="btn btn-colab"> Abrir en Colab</a>'
        btn_ver = f'<a href="{ruta_rel}" target="_blank" class="btn btn-ver">️ Ver contenido</a>'
        
        html += f"""
        <div class='nb-card'>
            <div class='nb-titulo'>{row['titulo']}</div>
            <div class='nb-bloque'>{row['bloque']}</div>
            <div class='nb-descripcion'>{row['descripcion'] if row['descripcion'] else 'Sin descripción disponible'}</div>
            <div class='nb-archivo'>📄 {row['archivo']}</div>
            {btn_colab}
            {btn_ver}
        </div>
        """
    
    display(HTML(html))

mostrar_notebooks()

# Celda 4: Filtros
print("FILTROS DISPONIBLES")
print("="*70)
print("\nBloques disponibles:")
for i, bloque in enumerate(['Todos'] + list(df_notebooks['bloque'].unique()), 1):
    count = len(df_notebooks[df_notebooks['bloque'] == bloque]) if bloque != 'Todos' else len(df_notebooks)
    print(f"  {i}. {bloque} ({count} notebooks)")

print("\nFunciones disponibles:")
print("  • mostrar_notebooks()")
print("  • mostrar_notebooks(filtro_bloque='Bloque_I_Comportamiento')")
print("  • mostrar_notebooks(filtro_busqueda='Momentum')")

# Celda 5: Tabla comparativa
def tabla_notebooks():
    print("TABLA DE NOTEBOOKS DEL PROYECTO")
    print("="*70)
    df_tabla = df_notebooks[['titulo', 'bloque', 'archivo']].copy()
    df_tabla.columns = ['Título', 'Bloque', 'Archivo']
    display(df_tabla)
    print(f"\nTotal: {len(df_notebooks)} notebooks")
    print(f"Bloques: {df_notebooks['bloque'].nunique()}")

tabla_notebooks()

# Celda 6: Estadísticas
def estadisticas_notebooks():
    print("ESTADÍSTICAS DE NOTEBOOKS")
    print("="*70)
    
    print("Distribución por bloque:")
    for bloque in df_notebooks['bloque'].unique():
        count = len(df_notebooks[df_notebooks['bloque'] == bloque])
        print(f"  • {bloque}: {count} notebooks")
    
    print(f"\nTotal de notebooks: {len(df_notebooks)}")
    print(f"Total de bloques: {df_notebooks['bloque'].nunique()}")
    
    total_celdas = 0
    total_codigo = 0
    total_markdown = 0
    
    for nb_info in lista_notebooks:
        try:
            with open(nb_info['ruta_completa'], 'r', encoding='utf-8') as f:
                nb = json.load(f)
            celdas = nb.get('cells', [])
            total_celdas += len(celdas)
            total_codigo += sum(1 for c in celdas if c.get('cell_type') == 'code')
            total_markdown += sum(1 for c in celdas if c.get('cell_type') == 'markdown')
        except:
            pass
    
    print(f"\nTotal de celdas en todos los notebooks: {total_celdas}")
    print(f"  • Celdas de código: {total_codigo}")
    print(f"  • Celdas de markdown: {total_markdown}")
    print("="*70)

estadisticas_notebooks()